In [ ]:
# =============================================================================
# Project:  lv-syn-grid
# Author:   Roman S.
# Created:  2025
#
# Description:
# A network graph is generated from OpenStreetMap data. A Minimum Spanning Tree
# is computed to derive a simplified radial topology, followed by a graph
# partitioning step. Each resulting partition represents a low-voltage feeder
# that is supplied by exactly one transformer.
#
# License: MIT License
#
# =============================================================================

# import for data/math/statistics
import pandas as pd
import geopandas as gp
import networkx as nx

# import for plotting
import matplotlib.pyplot as plt

# import for file i/o
import pickle

# import utils
from scripts import parameters as prm
from scripts import plot_utils as pu
from scripts import graph_utils as gu

In [ ]:
class GeoGraphModel:
    """
    Builds the graph model.
    """
    def __init__(self):
        self.__b_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILEPATH_BUILDINGS_PARQUET)
        self.__s_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILEPATH_STREETS_PARQUET)
        self.__n_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILEPATH_NODES_PARQUET)
        self.__n_gdf["node"] = "ND" + self.__n_gdf["node"].astype(str)
        self.__t_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILEPATH_TRANSFORMERS_PARQUET)
        self.__l_gdf = gp.read_parquet(prm.FILEPATH_DATA + prm.FILEPATH_LINES_PARQUET)
        
        self.__G_sg_df = pd.DataFrame(columns=['partition', 'subgraph_file', 'subgraph', 'associated_nodes', 'pos'])    # dataframe for subgraphs
        self.__H_sg_df = None
        
        self.__G_lv = None
        self.__H_lv = None
        self.__J_mv = None
        self.__H_s_lv = []

        self.__fig_H_s_lv = None  # define object for storing the figure
        self.__fig_G_lv = None    # define object for storing the figure
        self.__fig_H_lv = None    # define object for storing the figure


    def create_G_lv(self) -> None:
        """
        Creates the low-voltage network graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__G_lv = gu.create_G_lv(   l_gdf=self.__l_gdf,
                                        n_gdf=self.__n_gdf,
                                        b_gdf=self.__b_gdf,
                                        t_gdf=self.__t_gdf)
        
    
    def create_H_lv(self) -> None:
        """
        Creates the minimum spanning tree of the low-voltage network graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__H_lv = gu.apply_mst_algorithm(self.__G_lv)


    def create_H_sg_lv(self) -> None:
        """
        Partitions the low-voltage spanning tree into subgraphs.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__H_sg_df = gu.part_H(G=self.__H_lv, df=self.__G_sg_df)    # partition graph H_lv
        self.__H_s_lv = gu.extract_H_s(self.__H_sg_df, self.__H_lv)  # now all subgraphs of H_lv are extracted


    def rework_H_s_lv(self) -> None:
        """
        Reworks the low-voltage subgraphs by resizing transformer capacities.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        # resize transformer power according to the demands
        self.__H_s_lv = gu.separate_networks(G_=self.__H_s_lv, tr_syn_gdf=self.__t_gdf)
        self.__H_s_lv = gu.resize_network_transformers(self.__H_s_lv)
    

    def get_H_sg_df(self) -> pd.DataFrame:
        """
        Returns the DataFrame containing the subgraph partitioning information.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        pandas.DataFrame
            DataFrame with low-voltage subgraph assignments.
        """
        return self.__H_sg_df
    

    def get_H_s_lv(self) -> list:
        """
        Returns the list of low-voltage subgraphs.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        list
            List of low-voltage network subgraphs.
        """
        return self.__H_s_lv


    def get_G_lv(self) -> nx.MultiGraph:
        """
        Returns the low-voltage network graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        networkx.MultiGraph
            Low-voltage network graph.
        """
        return self.__G_lv
    

    def get_H_lv(self) -> nx.MultiGraph:
        """
        Returns the minimum spanning tree low-voltage network graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        networkx.MultiGraph
            Minimum spanning tree low-voltage network graph.
        """
        return self.__H_lv
    

    def __save_G_lv(self) -> None:
        """
        Saves the low-voltage network graph to GML file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        if self.__G_lv is not None:
            nx.write_gml(self.__G_lv, prm.FILEPATH_DATA + 'G_lv.gml')


    def __save_H_lv(self) -> None:
        """
        Saves the minimum spanning tree low-voltage network graph to GML file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        if self.__H_lv is not None:
            nx.write_gml(self.__H_lv, prm.FILEPATH_DATA + 'H_lv.gml')


    def __save_H_s_lv(self) -> None:
        """
        Saves the partitioned minimum spanning tree low-voltage network graph to GML file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        if self.__H_s_lv is not None:
            with open(prm.FILEPATH_DATA + 'H_s_lv.pkl', 'wb') as f:
                pickle.dump(self.__H_s_lv, f)


    def load_H_s_lv(self) -> None:
        """
        Loads the low-voltage subgraphs from a Pickle file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        with open(prm.FILEPATH_DATA + 'H_s_lv.pkl', 'rb') as f:
            self.__H_s_lv = pickle.load(f)


    def load_H_lv(self) -> None:
        """
        Loads the low-voltage minimum spanning tree graph from a Pickle file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__H_lv = nx.read_gml(prm.FILEPATH_DATA + 'H_lv.gml')

    
    def load_G_lv(self) -> None:
        """
        Loads the low-voltage graph from a Pickle file.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__G_lv = nx.read_gml(prm.FILEPATH_DATA + 'G_lv.gml')


    def save_graph_model(self) -> None:
        """
        Saves all graph model components to files.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__save_G_lv()
        self.__save_H_lv()
        self.__save_H_s_lv()

    
    def plot_H_s_lv(self) -> None:
        """
        Plots the low-voltage subgraphs.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """
        self.__fig_H_s_lv = pu.plot_H_s_lv( Hs=self.__H_s_lv,
                                            s_gdf=self.__s_gdf,
                                            figsize=[9, 9])
        
    
    def plot_G_lv(self) -> None:
        """
        Plots the low-voltage graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """ 
        self.__fig_G_lv = pu.plot_G_lv( G=self.__G_lv,
                                        s_gdf=self.__s_gdf,
                                        figsize=[9, 9],
                                        color='black')
        

    def plot_H_lv(self) -> None:
        """
        Plots the low-voltage minimum spanning tree graph.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """    
        self.__fig_H_lv = pu.plot_G_lv( G=self.__H_lv,
                                        s_gdf=self.__s_gdf,
                                        figsize=[9, 9],
                                        color='gray')
        
    
    def get_fig_H_s_lv(self) -> plt.figure:
        """
        Returns the figure of the low-voltage subgraph plot.
        
        Parameters:
        -----------
        None

        Returns:
        --------
        None
        """    
        return self.__fig_H_s_lv
    

    def save_fig_G_lv(self, filename: str) -> None:
        """
        Saves the low-voltage network graph to figures.
        
        Parameters:
        -----------
        filename : str
            Base filename used for saving the figure.

        Returns:
        --------
        None
        """
        self.__fig_G_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.svg', format='svg', dpi=600, bbox_inches='tight', transparent=True)
        self.__fig_G_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.png', format='png', dpi=600, bbox_inches='tight', transparent=True)


    def save_fig_H_lv(self, filename: str) -> None:
        """
        Saves the low-voltage network minimum spanning tree graph to figures.
        
        Parameters:
        -----------
        filename : str
            Base filename used for saving the figure.

        Returns:
        --------
        None
        """
        self.__fig_H_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.svg', format='svg', dpi=600, bbox_inches='tight', transparent=True)
        self.__fig_H_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.png', format='png', dpi=600, bbox_inches='tight', transparent=True)


    def save_fig_H_s_lv(self, filename: str) -> None:
        """
        Saves the low-voltage network subgraphs to figures.
        
        Parameters:
        -----------
        filename : str
            Base filename used for saving the figure.

        Returns:
        --------
        None
        """
        self.__fig_H_s_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.svg', format='svg', dpi=600, bbox_inches='tight', transparent=True)
        self.__fig_H_s_lv.savefig(f'{prm.FILEPATH_PLOTS}{filename}.png', format='png', dpi=600, bbox_inches='tight', transparent=True)

In [ ]:
ggm = GeoGraphModel()

In [ ]:
#ggm.create_G_lv()
#ggm.create_H_lv()
#ggm.rework_H_s_lv()

#ggm.plot_H_s_lv()

In [ ]:
# load already saved graph models and store it into objects
ggm.load_G_lv()
ggm.load_H_lv()
ggm.load_H_s_lv()

In [ ]:
ggm.plot_G_lv()

In [ ]:
ggm.plot_H_lv()

In [ ]:
ggm.plot_H_s_lv()

In [ ]:
# save the graph models to *.gml files
ggm.save_graph_model()

In [ ]:
# save the created figure of the graph model to *.svg and *.png files
ggm.save_fig_G_lv(filename='G_lv')
ggm.save_fig_H_lv(filename='H_lv')
ggm.save_fig_H_s_lv(filename='H_s_lv')